# Notebook 6: The Structure Module and Invariant Point Attention

**Objective:** Understand the Structure Module -- how AlphaFold2 converts abstract representations into 3D atomic coordinates using Invariant Point Attention (IPA) and SE(3)-equivariant operations.

---

In the preceding notebooks we traced how AlphaFold2 builds rich residue-level and pair-level representations through the Evoformer stack. Those representations, however, live in an abstract feature space -- they carry no explicit 3D geometry. The **Structure Module** is the component that bridges the gap between learned representations and physical atomic coordinates. Its design is built on two pillars:

1. **Invariant Point Attention (IPA):** an attention mechanism that is simultaneously aware of 3D spatial relationships and invariant to global rigid-body transformations.
2. **SE(3)-equivariant frame updates:** an iterative refinement loop that updates per-residue rigid-body frames while guaranteeing that a global rotation/translation of the input produces a correspondingly rotated/translated output.

We will develop these ideas from first principles, implement simplified versions in NumPy, and visualize every stage.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

---
## 1. From Representations to 3D Coordinates

The Evoformer produces two key outputs:

- The **single representation** $\mathbf{s}_i \in \mathbb{R}^{c_s}$ for each residue $i$, extracted as the first row of the refined MSA representation.
- The **pair representation** $\mathbf{z}_{ij} \in \mathbb{R}^{c_z}$ encoding pairwise relationships.

Neither of these contains explicit 3D coordinates. The Structure Module takes $\{\mathbf{s}_i\}$ and $\{\mathbf{z}_{ij}\}$ as input and outputs a set of **backbone frames**:

$$T_i = (\mathbf{R}_i,\, \mathbf{t}_i), \qquad \mathbf{R}_i \in SO(3),\; \mathbf{t}_i \in \mathbb{R}^3$$

for each residue $i = 1, \ldots, N_{\text{res}}$. From these frames, together with predicted torsion angles, all heavy-atom coordinates are reconstructed.

In [ ]:
# ── Conceptual diagram: abstract representations --> Structure Module --> 3D backbone ──

fig = plt.figure(figsize=(14, 5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 0.4, 1.2], wspace=0.05)

# --- Left panel: abstract representation (heatmap-like) ---
ax0 = fig.add_subplot(gs[0])
np.random.seed(7)
N_demo = 12
c_s_demo = 8
single_rep = np.random.randn(N_demo, c_s_demo)
pair_rep = np.random.randn(N_demo, N_demo)
pair_rep = (pair_rep + pair_rep.T) / 2

# Show single representation as a small heatmap
im = ax0.imshow(single_rep, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax0.set_xlabel('Feature dimension $c_s$', fontsize=12)
ax0.set_ylabel('Residue $i$', fontsize=12)
ax0.set_title('Single representation $\mathbf{s}_i$', fontsize=13)
ax0.tick_params(labelsize=11)

# --- Middle panel: arrow ---
ax1 = fig.add_subplot(gs[1])
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.annotate('', xy=(0.85, 0.5), xytext=(0.15, 0.5),
             arrowprops=dict(arrowstyle='->', lw=2.5, color='#333333'))
ax1.text(0.5, 0.62, 'Structure\nModule', ha='center', va='bottom', fontsize=13,
         style='italic', color='#333333')
ax1.axis('off')

# --- Right panel: 3D backbone ---
ax2 = fig.add_subplot(gs[2], projection='3d')

# Generate a helix-like backbone
t_helix = np.linspace(0, 3 * np.pi, N_demo)
x_helix = 3.0 * np.cos(t_helix)
y_helix = 3.0 * np.sin(t_helix)
z_helix = t_helix * 1.5

ax2.plot(x_helix, y_helix, z_helix, '-o', color='#2c7fb8', lw=2, markersize=6,
         markerfacecolor='#d95f02', zorder=5)
for idx in range(N_demo):
    ax2.text(x_helix[idx] + 0.3, y_helix[idx] + 0.3, z_helix[idx] + 0.2,
             f'{idx}', fontsize=9, color='#555555')

ax2.set_xlabel('x', fontsize=11)
ax2.set_ylabel('y', fontsize=11)
ax2.set_zlabel('z', fontsize=11)
ax2.set_title('Backbone frames $T_i = (\mathbf{R}_i, \mathbf{t}_i)$', fontsize=13)
ax2.tick_params(labelsize=9)
ax2.view_init(elev=20, azim=135)

fig.suptitle('The Structure Module: from abstract features to 3D coordinates',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Rigid Body Frames and SE(3)

The backbone of each residue is described by a **rigid-body frame** consisting of a rotation and a translation:

$$T = (\mathbf{R},\, \mathbf{t}), \qquad \mathbf{R} \in SO(3),\; \mathbf{t} \in \mathbb{R}^3.$$

The group $SE(3) = SO(3) \ltimes \mathbb{R}^3$ is the **special Euclidean group** in three dimensions -- the group of all rigid-body motions (rotations and translations). Its operations are:

**Frame application** (acting on a point $\mathbf{x} \in \mathbb{R}^3$):
$$T \circ \mathbf{x} = \mathbf{R}\mathbf{x} + \mathbf{t}$$

**Frame composition:**
$$T_1 \circ T_2 = (\mathbf{R}_1 \mathbf{R}_2,\; \mathbf{R}_1 \mathbf{t}_2 + \mathbf{t}_1)$$

**Frame inverse:**
$$T^{-1} = (\mathbf{R}^\top,\; -\mathbf{R}^\top \mathbf{t})$$

One can verify $T \circ T^{-1} = T^{-1} \circ T = (\mathbf{I},\, \mathbf{0})$, confirming the group structure.

In [ ]:
class Frame:
    """Rigid-body frame T = (R, t) in SE(3)."""
    
    def __init__(self, R, t):
        """
        Parameters
        ----------
        R : ndarray, shape (3, 3) -- rotation matrix in SO(3)
        t : ndarray, shape (3,)   -- translation vector
        """
        self.R = np.array(R, dtype=np.float64)
        self.t = np.array(t, dtype=np.float64)
    
    def apply(self, x):
        """Apply frame to point(s): T . x = Rx + t."""
        return self.R @ x + self.t
    
    def compose(self, other):
        """Compose: self . other = (R1 R2, R1 t2 + t1)."""
        R_new = self.R @ other.R
        t_new = self.R @ other.t + self.t
        return Frame(R_new, t_new)
    
    def inverse(self):
        """Inverse: T^{-1} = (R^T, -R^T t)."""
        R_inv = self.R.T
        t_inv = -R_inv @ self.t
        return Frame(R_inv, t_inv)
    
    @staticmethod
    def identity():
        return Frame(np.eye(3), np.zeros(3))
    
    def __repr__(self):
        return f"Frame(R=\n{self.R},\nt={self.t})"


def rotation_matrix(axis, angle):
    """Rodrigues' formula: rotation by `angle` radians about `axis`."""
    axis = np.asarray(axis, dtype=np.float64)
    axis = axis / np.linalg.norm(axis)
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)


# Create two frames
R1 = rotation_matrix([0, 0, 1], np.pi / 4)   # 45 deg about z
t1 = np.array([2.0, 0.0, 0.0])
T1 = Frame(R1, t1)

R2 = rotation_matrix([0, 1, 0], np.pi / 3)   # 60 deg about y
t2 = np.array([0.0, 1.5, 1.0])
T2 = Frame(R2, t2)

# Composition and inverse
T12 = T1.compose(T2)
T1_inv = T1.inverse()

# Verify inverse
T_check = T1.compose(T1_inv)
print("T1 . T1^{-1} rotation (should be I):")
print(np.round(T_check.R, 10))
print("T1 . T1^{-1} translation (should be 0):")
print(np.round(T_check.t, 10))

In [ ]:
# ── Visualize frames as coordinate axes in 3D ──

def draw_frame(ax, frame, label, colors=('r', 'g', 'b'), length=1.0, lw=2.0):
    """Draw a coordinate frame as three quiver arrows."""
    origin = frame.t
    for k, (c, name) in enumerate(zip(colors, ['x', 'y', 'z'])):
        direction = frame.R[:, k] * length
        ax.quiver(*origin, *direction, color=c, arrow_length_ratio=0.12, lw=lw)
    ax.text(origin[0], origin[1], origin[2] - 0.35, label, fontsize=12,
            ha='center', color='#333333')

fig = plt.figure(figsize=(14, 6))

# --- Panel (a): T1 and T2 ---
ax_a = fig.add_subplot(121, projection='3d')
draw_frame(ax_a, Frame.identity(), '$I$', length=0.8, lw=1.5)
draw_frame(ax_a, T1, '$T_1$')
draw_frame(ax_a, T2, '$T_2$')
ax_a.set_title('Individual frames $T_1$ and $T_2$', fontsize=13)
ax_a.set_xlabel('x', fontsize=11); ax_a.set_ylabel('y', fontsize=11); ax_a.set_zlabel('z', fontsize=11)
ax_a.set_xlim(-1, 4); ax_a.set_ylim(-1, 3); ax_a.set_zlim(-1, 3)
ax_a.view_init(elev=20, azim=135)
ax_a.tick_params(labelsize=9)

# --- Panel (b): composition T1.T2 and inverse T1^{-1} ---
ax_b = fig.add_subplot(122, projection='3d')
draw_frame(ax_b, Frame.identity(), '$I$', length=0.8, lw=1.5)
draw_frame(ax_b, T12, '$T_1 \circ T_2$')
draw_frame(ax_b, T1_inv, '$T_1^{-1}$')
ax_b.set_title('Composition $T_1 \\circ T_2$ and inverse $T_1^{-1}$', fontsize=13)
ax_b.set_xlabel('x', fontsize=11); ax_b.set_ylabel('y', fontsize=11); ax_b.set_zlabel('z', fontsize=11)
ax_b.set_xlim(-2.5, 3.5); ax_b.set_ylim(-1, 3.5); ax_b.set_zlim(-1, 3)
ax_b.view_init(elev=20, azim=135)
ax_b.tick_params(labelsize=9)

# Legend
patches = [mpatches.Patch(color='r', label='x-axis'),
           mpatches.Patch(color='g', label='y-axis'),
           mpatches.Patch(color='b', label='z-axis')]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=11,
           frameon=True, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.show()

---
## 3. Why Equivariance Matters

A function $f$ mapping residue frames to residue frames is **SE(3)-equivariant** if, for any global rigid-body transformation $T_{\text{global}} \in SE(3)$:

$$f\bigl(T_{\text{global}} \circ T_i\bigr) = T_{\text{global}} \circ f(T_i), \qquad \forall\, i.$$

In words: rotating and translating the entire input structure, then running the Structure Module, gives the same result as first running the Structure Module and then applying the rotation and translation to the output.

This property is critical because **the laws of physics are SE(3)-invariant** -- a protein does not change its fold depending on its orientation in space. AlphaFold2 guarantees equivariance by construction: all operations within the Structure Module are expressed in local, frame-relative coordinates. The only place where global coordinates appear is when transforming local points into a shared frame for distance computation, and those distances are invariant under global transformations.

In [ ]:
# ── Demonstrate SE(3)-equivariance ──

np.random.seed(10)
N_res = 8

# Create synthetic residue frames along a helix
frames_original = []
for i in range(N_res):
    angle_z = i * 0.6
    R_i = rotation_matrix([0, 0, 1], angle_z) @ rotation_matrix([1, 0, 0], i * 0.3)
    t_i = np.array([3.0 * np.cos(angle_z), 3.0 * np.sin(angle_z), i * 1.5])
    frames_original.append(Frame(R_i, t_i))

# A simple "structure module" simulation: small local perturbation (frame-relative)
def simulated_structure_module(frames):
    """Apply a deterministic, frame-relative update to each frame."""
    updated = []
    for i, T in enumerate(frames):
        # Small local rotation and translation (same for each residue index)
        rng = np.random.RandomState(seed=i + 100)  # deterministic per residue
        dR = rotation_matrix(rng.randn(3), 0.15)
        dt = rng.randn(3) * 0.3
        delta = Frame(dR, dt)
        updated.append(T.compose(delta))  # local update in T's frame
    return updated

# Apply structure module to original
frames_out = simulated_structure_module(frames_original)

# Apply a random global transformation
R_global = rotation_matrix([1, 1, 0], np.pi / 5)
t_global = np.array([5.0, -3.0, 2.0])
T_global = Frame(R_global, t_global)

# Path A: transform input, then apply structure module
frames_rotated_input = [T_global.compose(T) for T in frames_original]
frames_pathA = simulated_structure_module(frames_rotated_input)

# Path B: apply structure module to original, then transform output
frames_pathB = [T_global.compose(T) for T in frames_out]

# Check: Path A and Path B should give the same result
max_rot_err = max(np.max(np.abs(a.R - b.R)) for a, b in zip(frames_pathA, frames_pathB))
max_trans_err = max(np.max(np.abs(a.t - b.t)) for a, b in zip(frames_pathA, frames_pathB))
print(f"Max rotation error between Path A and Path B:    {max_rot_err:.2e}")
print(f"Max translation error between Path A and Path B: {max_trans_err:.2e}")
print("Equivariance verified!" if max_rot_err < 1e-12 and max_trans_err < 1e-12 else "ERROR")

In [ ]:
# ── Visualize both paths in 3D ──

def extract_positions(frames):
    return np.array([T.t for T in frames])

pos_orig = extract_positions(frames_original)
pos_pathA = extract_positions(frames_pathA)
pos_pathB = extract_positions(frames_pathB)

fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot(*pos_orig.T, 'o-', color='#2c7fb8', lw=2, markersize=6, label='Original input')
pos_out_orig = extract_positions(frames_out)
ax1.plot(*pos_out_orig.T, 's-', color='#d95f02', lw=2, markersize=6, label='Output $f(T_i)$')
ax1.set_title('Original: input and output', fontsize=13)
ax1.legend(fontsize=11)
ax1.set_xlabel('x', fontsize=11); ax1.set_ylabel('y', fontsize=11); ax1.set_zlabel('z', fontsize=11)
ax1.view_init(elev=20, azim=135)
ax1.tick_params(labelsize=9)

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot(*pos_pathA.T, '^-', color='#e7298a', lw=2, markersize=7,
         label='Path A: $f(T_{\mathrm{global}} \circ T_i)$')
ax2.plot(*pos_pathB.T, 'D--', color='#1b9e77', lw=2, markersize=6, alpha=0.8,
         label='Path B: $T_{\mathrm{global}} \circ f(T_i)$')
ax2.set_title('Equivariance check: both paths agree', fontsize=13)
ax2.legend(fontsize=11)
ax2.set_xlabel('x', fontsize=11); ax2.set_ylabel('y', fontsize=11); ax2.set_zlabel('z', fontsize=11)
ax2.view_init(elev=20, azim=135)
ax2.tick_params(labelsize=9)

plt.tight_layout()
plt.show()

---
## 4. Invariant Point Attention (IPA)

IPA is the core attention mechanism inside the Structure Module. It fuses three sources of information:

### (a) Standard QKV attention on the single representation

$$q_i = W^Q \mathbf{s}_i, \quad k_j = W^K \mathbf{s}_j, \quad v_j = W^V \mathbf{s}_j$$

$$a_{ij}^{\text{seq}} = \frac{1}{\sqrt{d_h}} \, q_i^\top k_j$$

### (b) Pair bias

$$b_{ij} = \text{Linear}(\mathbf{z}_{ij})$$

A scalar bias read directly from the pair representation, injecting co-evolutionary and distance information.

### (c) Point attention (the novel contribution)

For each head $h$ and each of $N_p$ query/key points:

$$\mathbf{p}_i^{q,h,p} = T_i \circ \text{Linear}^{q,h,p}(\mathbf{s}_i) \qquad \text{(query points projected into global frame)}$$
$$\mathbf{p}_j^{k,h,p} = T_j \circ \text{Linear}^{k,h,p}(\mathbf{s}_j) \qquad \text{(key points projected into global frame)}$$

The point attention logit is:

$$a_{ij}^{\text{point}} = -\frac{\gamma_h}{2} \sum_{p=1}^{N_p} \left\| \mathbf{p}_i^{q,h,p} - \mathbf{p}_j^{k,h,p} \right\|^2$$

where $\gamma_h > 0$ is a learned, head-specific weight (initialized so that point attention has a soft length scale of about $\sim$ 9 \AA).

### Combined attention

$$\alpha_{ij} = \text{softmax}_j \!\left( a_{ij}^{\text{seq}} + b_{ij} + a_{ij}^{\text{point}} \right)$$

The key insight: **because points are projected from local frames into a shared global frame, the squared distances $\|\mathbf{p}_i^q - \mathbf{p}_j^k\|^2$ are invariant to global rigid-body transformations.** Applying a global $T_{\text{global}}$ shifts both points equally, leaving their difference unchanged.

In [ ]:
# ── Simplified IPA implementation ──

np.random.seed(42)

N_res = 10
c_s = 16    # single representation dim
c_z = 8     # pair representation dim
d_h = 8     # head dim for QKV
N_points = 3  # number of query/key points per head

# Generate helical residue frames
frames_ipa = []
for i in range(N_res):
    theta = i * 100.0 * np.pi / 180.0  # ~100 deg per residue (alpha helix)
    R_i = rotation_matrix([0, 0, 1], theta)
    t_i = np.array([3.8 * np.cos(theta), 3.8 * np.sin(theta), i * 1.5])
    frames_ipa.append(Frame(R_i, t_i))

# Synthetic representations
S = np.random.randn(N_res, c_s) * 0.5   # single representation
Z = np.random.randn(N_res, N_res, c_z) * 0.3  # pair representation
Z = (Z + Z.transpose(1, 0, 2)) / 2  # symmetrize

# ---- (a) Standard QKV attention ----
W_Q = np.random.randn(c_s, d_h) * 0.1
W_K = np.random.randn(c_s, d_h) * 0.1

Q = S @ W_Q  # (N_res, d_h)
K = S @ W_K  # (N_res, d_h)

attn_seq = (Q @ K.T) / np.sqrt(d_h)  # (N_res, N_res)

# ---- (b) Pair bias ----
W_pair = np.random.randn(c_z, 1) * 0.1
bias = (Z @ W_pair).squeeze(-1)  # (N_res, N_res)

# ---- (c) Point attention ----
# Project local points from single representation
W_q_pts = np.random.randn(c_s, N_points * 3) * 0.1
W_k_pts = np.random.randn(c_s, N_points * 3) * 0.1

q_local = (S @ W_q_pts).reshape(N_res, N_points, 3)  # local query points
k_local = (S @ W_k_pts).reshape(N_res, N_points, 3)  # local key points

# Transform to global frame
q_global = np.zeros_like(q_local)
k_global = np.zeros_like(k_local)
for i in range(N_res):
    for p in range(N_points):
        q_global[i, p] = frames_ipa[i].apply(q_local[i, p])
        k_global[i, p] = frames_ipa[i].apply(k_local[i, p])

# Compute squared distances
gamma = 1.0 / (2.0 * N_points)  # weight
attn_point = np.zeros((N_res, N_res))
for i in range(N_res):
    for j in range(N_res):
        dist_sq = np.sum((q_global[i] - k_global[j]) ** 2)
        attn_point[i, j] = -gamma * dist_sq

# ---- Combined attention ----
logits = attn_seq + bias + attn_point

# Softmax
def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

attn_weights = softmax(logits, axis=-1)

print(f"Attention matrix shape: {attn_weights.shape}")
print(f"Row sums (should be 1): {attn_weights.sum(axis=1).round(6)}")

In [ ]:
# ── Visualize the three attention components and combined map ──

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))

# Normalize each for display
maps = [
    ('(a) Sequence attn $a_{ij}^{\\mathrm{seq}}$', softmax(attn_seq)),
    ('(b) Pair bias $b_{ij}$ (softmax)', softmax(bias)),
    ('(c) Point attn $a_{ij}^{\\mathrm{point}}$ (softmax)', softmax(attn_point)),
    ('(d) Combined $\\alpha_{ij}$', attn_weights),
]

for ax, (title, mat) in zip(axes, maps):
    im = ax.imshow(mat, cmap='viridis', aspect='equal', vmin=0)
    ax.set_xlabel('Key residue $j$', fontsize=11)
    ax.set_ylabel('Query residue $i$', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.tick_params(labelsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# ── Show how 3D proximity modulates attention ──

# Compute pairwise CA distances
positions_ipa = np.array([T.t for T in frames_ipa])
dist_matrix = np.sqrt(np.sum((positions_ipa[:, None] - positions_ipa[None, :]) ** 2, axis=-1))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(dist_matrix, cmap='magma_r', aspect='equal')
axes[0].set_title('Pairwise $C_\\alpha$ distance (\u00c5)', fontsize=13)
axes[0].set_xlabel('Residue $j$', fontsize=11)
axes[0].set_ylabel('Residue $i$', fontsize=11)
axes[0].tick_params(labelsize=10)
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(attn_weights, cmap='viridis', aspect='equal', vmin=0)
axes[1].set_title('Combined IPA attention $\\alpha_{ij}$', fontsize=13)
axes[1].set_xlabel('Key residue $j$', fontsize=11)
axes[1].set_ylabel('Query residue $i$', fontsize=11)
axes[1].tick_params(labelsize=10)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.suptitle('3D proximity (left) modulates IPA attention (right):\nnearby residues receive higher attention',
             fontsize=13, y=1.04)
plt.tight_layout()
plt.show()

---
## 5. Backbone Update

After each IPA layer, the Structure Module updates the backbone frames. The IPA output (after a feed-forward network) predicts per-residue updates:

$$\Delta T_i = (\Delta \mathbf{R}_i,\, \Delta \mathbf{t}_i)$$

The new frame is obtained by **composing** the current frame with the local update:

$$T_i^{(l+1)} = T_i^{(l)} \circ (\Delta \mathbf{R}_i,\, \Delta \mathbf{t}_i)$$

This is the key to equivariance: the update is applied **in the residue's local frame**. A global rotation $T_{\text{global}}$ commutes through the composition:

$$(T_{\text{global}} \circ T_i^{(l)}) \circ \Delta T_i = T_{\text{global}} \circ (T_i^{(l)} \circ \Delta T_i) = T_{\text{global}} \circ T_i^{(l+1)}$$

**Initialization:** All frames begin at the identity $T_i^{(0)} = (\mathbf{I},\, \mathbf{0})$ -- i.e., all residues start at the origin. The structure emerges through iterative refinement.

In [ ]:
# ── Simulate 8 iterations of backbone refinement ──

np.random.seed(2024)
N_res_bb = 20
n_iters = 8

# Target: a helix shape (we'll guide the updates toward this)
target_positions = np.zeros((N_res_bb, 3))
for i in range(N_res_bb):
    theta = i * 100.0 * np.pi / 180.0
    target_positions[i] = [5.0 * np.cos(theta), 5.0 * np.sin(theta), i * 1.5]

# Initialize all frames at identity
frames_history = []  # list of lists (one per iteration)
current_frames = [Frame.identity() for _ in range(N_res_bb)]
frames_history.append([Frame(T.R.copy(), T.t.copy()) for T in current_frames])

# Iterative refinement: at each step, move a fraction toward the target
for iteration in range(n_iters):
    progress = (iteration + 1) / n_iters
    noise_scale = 0.4 * (1 - progress)  # decreasing noise
    
    new_frames = []
    for i in range(N_res_bb):
        # Compute desired global position at this stage
        desired_t = target_positions[i] * progress + np.random.randn(3) * noise_scale
        
        # Compute the local update that would move us toward the desired position
        T_current = current_frames[i]
        T_inv = T_current.inverse()
        
        # Desired in local frame
        delta_t_global = desired_t - T_current.t
        delta_t_local = T_inv.R @ delta_t_global
        
        # Small rotation toward target orientation
        angle_target = i * 100.0 * np.pi / 180.0
        R_target = rotation_matrix([0, 0, 1], angle_target * progress)
        dR = T_current.R.T @ R_target  # relative rotation in local frame
        
        delta_frame = Frame(dR, delta_t_local)
        new_frame = T_current.compose(delta_frame)
        new_frames.append(new_frame)
    
    current_frames = new_frames
    frames_history.append([Frame(T.R.copy(), T.t.copy()) for T in current_frames])

# Plot iterations 0, 2, 4, 6, 8
plot_iters = [0, 2, 4, 6, 8]
fig = plt.figure(figsize=(18, 4))

colors_iter = ['#bdbdbd', '#9ecae1', '#6baed6', '#3182bd', '#08519c']

for panel_idx, it in enumerate(plot_iters):
    ax = fig.add_subplot(1, 5, panel_idx + 1, projection='3d')
    pos = np.array([T.t for T in frames_history[it]])
    ax.plot(pos[:, 0], pos[:, 1], pos[:, 2], 'o-',
            color=colors_iter[panel_idx], lw=1.8, markersize=4)
    ax.set_title(f'Iteration {it}', fontsize=12)
    ax.set_xlim(-7, 7); ax.set_ylim(-7, 7); ax.set_zlim(-2, 32)
    ax.set_xlabel('x', fontsize=9); ax.set_ylabel('y', fontsize=9); ax.set_zlabel('z', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.view_init(elev=15, azim=135)

fig.suptitle('Backbone refinement: from collapsed (identity) to folded 3D structure',
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

---
## 6. Torsion Angle Prediction

The backbone frames define the positions and orientations of the peptide backbone ($N$--$C_\alpha$--$C$), but proteins also have **side chains** whose geometry is specified by torsion (dihedral) angles. For each residue, AlphaFold2 predicts:

- Backbone torsion angles: $\phi$, $\psi$ (and $\omega \approx 180^\circ$ for trans peptide bonds)
- Side chain torsion angles: $\chi_1, \chi_2, \chi_3, \chi_4$ (number depends on residue type)

### The $\boldsymbol{(\sin\theta, \cos\theta)}$ parameterization

Angles are periodic: $\theta$ and $\theta + 2\pi$ represent the same geometry. If we predicted angles directly, the network would face a discontinuity at $\pm\pi$. Instead, AlphaFold2 predicts each angle as a **(sin, cos) pair**:

$$(\sin\chi_k,\, \cos\chi_k) = \text{Linear}\bigl(\text{ReLU}(\text{Linear}(\mathbf{s}_i))\bigr)$$

The angle is recovered as $\chi_k = \text{atan2}(\sin\chi_k, \cos\chi_k)$, and the output is normalized to lie on the unit circle. This parameterization is smooth and avoids all discontinuities.

In [ ]:
# ── Visualize torsion angle prediction: unit circle representation ──

np.random.seed(99)

fig = plt.figure(figsize=(14, 5.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1.3, 1], wspace=0.35)

# --- Left: residue with backbone + side chain torsions ---
ax_mol = fig.add_subplot(gs[0])

# Backbone atoms (simplified 2D projection)
bb_atoms = {
    'N':  (0.5, 3.0),
    'CA': (2.0, 2.5),
    'C':  (3.5, 3.0),
    'O':  (4.2, 4.0),
}
# Side chain atoms
sc_atoms = {
    'CB': (2.0, 1.2),
    'CG': (1.0, 0.3),
    'CD1': (0.0, -0.5),
    'CD2': (2.0, -0.5),
}

# Draw backbone bonds
bb_bonds = [('N', 'CA'), ('CA', 'C'), ('C', 'O')]
for a1, a2 in bb_bonds:
    p1, p2 = bb_atoms[a1], bb_atoms[a2]
    ax_mol.plot([p1[0], p2[0]], [p1[1], p2[1]], 'k-', lw=2.5)

# Draw side chain bonds
sc_bonds = [('CA', 'CB'), ('CB', 'CG'), ('CG', 'CD1'), ('CG', 'CD2')]
for a1, a2 in sc_bonds:
    atoms = {**bb_atoms, **sc_atoms}
    p1, p2 = atoms[a1], atoms[a2]
    ax_mol.plot([p1[0], p2[0]], [p1[1], p2[1]], '-', color='#e6550d', lw=2.5)

# Draw atoms
for name, pos in bb_atoms.items():
    color = '#2171b5' if name != 'O' else '#cb181d'
    ax_mol.plot(*pos, 'o', color=color, markersize=14, zorder=5)
    ax_mol.text(pos[0], pos[1] + 0.25, name, ha='center', fontsize=11, color='white',
               zorder=6, bbox=dict(boxstyle='round,pad=0.15', facecolor=color, alpha=0.9))

for name, pos in sc_atoms.items():
    ax_mol.plot(*pos, 'o', color='#e6550d', markersize=12, zorder=5)
    ax_mol.text(pos[0], pos[1] + 0.25, name, ha='center', fontsize=10, color='white',
               zorder=6, bbox=dict(boxstyle='round,pad=0.15', facecolor='#e6550d', alpha=0.9))

# Annotate torsion angles
ax_mol.annotate('$\\phi$', xy=(1.25, 2.75), fontsize=14, color='#2171b5')
ax_mol.annotate('$\\psi$', xy=(2.75, 2.75), fontsize=14, color='#2171b5')
ax_mol.annotate('$\\chi_1$', xy=(2.2, 1.8), fontsize=14, color='#e6550d')
ax_mol.annotate('$\\chi_2$', xy=(1.2, 0.7), fontsize=14, color='#e6550d')

# Draw curved arrows for torsion angles
for center, radius, theta_range, color in [
    ((2.0, 2.5), 0.45, (210, 310), '#2171b5'),  # phi at CA
    ((2.0, 2.5), 0.55, (30, 130), '#2171b5'),    # psi at CA  
    ((2.0, 1.2), 0.35, (200, 300), '#e6550d'),   # chi1 at CB
    ((1.0, 0.3), 0.35, (250, 350), '#e6550d'),   # chi2 at CG
]:
    arc_angles = np.linspace(np.radians(theta_range[0]), np.radians(theta_range[1]), 30)
    arc_x = center[0] + radius * np.cos(arc_angles)
    arc_y = center[1] + radius * np.sin(arc_angles)
    ax_mol.plot(arc_x, arc_y, '-', color=color, lw=1.5, alpha=0.7)

ax_mol.set_xlim(-1.0, 5.0)
ax_mol.set_ylim(-1.5, 4.8)
ax_mol.set_aspect('equal')
ax_mol.set_title('Residue with backbone and side chain torsions', fontsize=13)
ax_mol.axis('off')

# --- Right: unit circle representation ---
ax_circ = fig.add_subplot(gs[1])

# Draw unit circle
theta_circle = np.linspace(0, 2 * np.pi, 200)
ax_circ.plot(np.cos(theta_circle), np.sin(theta_circle), 'k-', lw=1.5, alpha=0.3)
ax_circ.axhline(0, color='grey', lw=0.5, alpha=0.5)
ax_circ.axvline(0, color='grey', lw=0.5, alpha=0.5)

# Plot predicted angles
angle_names = ['$\\phi$', '$\\psi$', '$\\chi_1$', '$\\chi_2$']
angle_values = [-1.2, 2.1, -0.8, 1.5]  # radians
colors_angle = ['#2171b5', '#4292c6', '#e6550d', '#fd8d3c']

for name, angle, color in zip(angle_names, angle_values, colors_angle):
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    ax_circ.plot([0, cos_a], [0, sin_a], '-', color=color, lw=2)
    ax_circ.plot(cos_a, sin_a, 'o', color=color, markersize=10, zorder=5)
    offset = 0.15
    ax_circ.text(cos_a * (1 + offset), sin_a * (1 + offset), name,
                fontsize=13, color=color, ha='center', va='center')

ax_circ.set_xlabel('$\\cos\\theta$', fontsize=12)
ax_circ.set_ylabel('$\\sin\\theta$', fontsize=12)
ax_circ.set_title('Unit circle: $(\\sin\\theta, \\cos\\theta)$\navoids discontinuities', fontsize=13)
ax_circ.set_xlim(-1.6, 1.6)
ax_circ.set_ylim(-1.6, 1.6)
ax_circ.set_aspect('equal')
ax_circ.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

---
## 7. All-Atom Coordinate Generation

Once we have backbone frames $T_i$ and torsion angles $\{\chi_k\}$ for each residue, we can reconstruct **every heavy atom** using ideal (library) bond lengths and angles.

Each atom's position is obtained by composing a chain of rigid-body transformations along the kinematic tree of the residue:

$$\mathbf{x}_{\text{atom}} = T_{\text{residue}} \circ T_{\chi_1} \circ T_{\chi_2} \circ \cdots \circ \mathbf{x}_{\text{ideal}}$$

where:
- $T_{\text{residue}} = (\mathbf{R}_i, \mathbf{t}_i)$ is the backbone frame for residue $i$.
- $T_{\chi_k}$ is the rigid-body transformation corresponding to a rotation by $\chi_k$ about the appropriate bond axis, with translation along the bond.
- $\mathbf{x}_{\text{ideal}}$ is the atom's position in the idealized local frame (from a residue geometry library).

This construction is purely geometric -- no learning is needed once the frames and torsion angles are known.

In [ ]:
# ── All-atom generation for a leucine residue ──

np.random.seed(55)

# Ideal bond lengths and angles (approximate, angstroms)
# We'll build a simplified leucine: N - CA - C(=O) backbone, then CB - CG - CD1, CD2

# Backbone frame for this residue
T_res = Frame(rotation_matrix([0, 1, 0], 0.3), np.array([5.0, 2.0, 3.0]))

# Torsion angles (radians)
chi1 = -1.1  # CA-CB
chi2 = 1.7   # CB-CG

# Build atoms step by step using frame composition
# Backbone atoms in local frame
N_local = np.array([-1.458, 0.0, 0.0])     # N atom
CA_local = np.array([0.0, 0.0, 0.0])        # CA at origin of local frame
C_local = np.array([1.524, 0.0, 0.0])        # C atom
O_local = np.array([2.153, 1.062, 0.0])      # O atom

# Side chain: CB along a rotated direction from CA
# chi1 rotation about the N-CA-CB plane
T_chi1 = Frame(rotation_matrix([1, 0, 0], chi1), np.array([0.0, 0.0, 0.0]))
CB_base = np.array([0.523, -0.774, -1.207])  # ideal CB offset from CA
CB_local = T_chi1.apply(CB_base)

# CG from CB: apply chi2 rotation
bond_CB_CG = np.array([0.0, -1.536, 0.0])  # CG offset from CB in CB frame
T_chi2 = Frame(rotation_matrix([0, 1, 0], chi2), CB_local)
CG_local = T_chi2.apply(bond_CB_CG)

# CD1 and CD2 branch from CG
CD1_offset = np.array([-1.0, -0.8, 0.6])
CD2_offset = np.array([1.0, -0.8, -0.6])
CD1_local = CG_local + CD1_offset
CD2_local = CG_local + CD2_offset

# Transform all atoms to global frame
atoms_local = {
    'N': N_local, 'CA': CA_local, 'C': C_local, 'O': O_local,
    'CB': CB_local, 'CG': CG_local, 'CD1': CD1_local, 'CD2': CD2_local
}
atoms_global = {name: T_res.apply(pos) for name, pos in atoms_local.items()}

# ── Plot ──
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Bonds
backbone_bonds = [('N', 'CA'), ('CA', 'C'), ('C', 'O')]
sidechain_bonds = [('CA', 'CB'), ('CB', 'CG'), ('CG', 'CD1'), ('CG', 'CD2')]

for a1, a2 in backbone_bonds:
    p1, p2 = atoms_global[a1], atoms_global[a2]
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
            '-', color='#2171b5', lw=3)

for a1, a2 in sidechain_bonds:
    p1, p2 = atoms_global[a1], atoms_global[a2]
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
            '-', color='#e6550d', lw=3)

# Atoms
bb_color_map = {'N': '#4292c6', 'CA': '#2171b5', 'C': '#2171b5', 'O': '#cb181d'}
for name in ['N', 'CA', 'C', 'O']:
    pos = atoms_global[name]
    ax.scatter(*pos, s=200, color=bb_color_map[name], zorder=5, edgecolors='k', linewidths=0.5)
    ax.text(pos[0] + 0.15, pos[1] + 0.15, pos[2] + 0.15, name, fontsize=12, color='#333333')

for name in ['CB', 'CG', 'CD1', 'CD2']:
    pos = atoms_global[name]
    ax.scatter(*pos, s=180, color='#e6550d', zorder=5, edgecolors='k', linewidths=0.5)
    ax.text(pos[0] + 0.15, pos[1] + 0.15, pos[2] + 0.15, name, fontsize=11, color='#333333')

# Draw backbone frame axes
draw_frame(ax, T_res, '$T_{\mathrm{res}}$', length=1.2, lw=1.5)

# Annotate chi angles
mid_chi1 = (atoms_global['CA'] + atoms_global['CB']) / 2
ax.text(mid_chi1[0] + 0.4, mid_chi1[1] - 0.3, mid_chi1[2],
        f'$\\chi_1 = {np.degrees(chi1):.0f}^\\circ$', fontsize=12, color='#e6550d')
mid_chi2 = (atoms_global['CB'] + atoms_global['CG']) / 2
ax.text(mid_chi2[0] + 0.4, mid_chi2[1] - 0.3, mid_chi2[2],
        f'$\\chi_2 = {np.degrees(chi2):.0f}^\\circ$', fontsize=12, color='#e6550d')

ax.set_xlabel('x (\u00c5)', fontsize=11)
ax.set_ylabel('y (\u00c5)', fontsize=11)
ax.set_zlabel('z (\u00c5)', fontsize=11)
ax.set_title('Leucine: all-atom reconstruction from backbone frame + torsion angles',
             fontsize=13)
ax.tick_params(labelsize=9)

# Legend
bb_patch = mpatches.Patch(color='#2171b5', label='Backbone atoms')
sc_patch = mpatches.Patch(color='#e6550d', label='Side chain atoms')
ax.legend(handles=[bb_patch, sc_patch], fontsize=11, loc='upper left')

ax.view_init(elev=20, azim=145)
plt.tight_layout()
plt.show()

---
## 8. The Full Structure Module Pipeline

The Structure Module runs for **8 iterations** (called "layers" in the AlphaFold2 paper), each consisting of:

1. **Invariant Point Attention (IPA):** updates single representations using 3D-aware attention.
2. **Transition (feed-forward):** a small MLP applied to each residue's updated representation.
3. **Backbone update:** predicts $\Delta T_i$ and composes with current frames.
4. **Torsion angle prediction:** predicts $(\sin, \cos)$ pairs for all torsion angles.

Crucially, **all 8 iterations share the same weights** (weight sharing, analogous to the recycling mechanism at the outer level). The single representation $\mathbf{s}_i$ is updated at each iteration, but the pair representation $\mathbf{z}_{ij}$ is fixed (taken from the Evoformer output).

The iterative refinement is conceptually similar to an optimization loop: starting from a collapsed initial state (all frames at the origin), the Structure Module progressively "unfolds" the protein into its 3D shape.

In [ ]:
# ── Flow diagram of the Structure Module pipeline ──

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

# Color scheme
c_input = '#c6dbef'
c_ipa = '#fd8d3c'
c_ffn = '#74c476'
c_bb = '#9e9ac8'
c_torsion = '#fb6a4a'
c_output = '#bcbddc'
c_loop = '#636363'

box_props = dict(boxstyle='round,pad=0.5', linewidth=1.5)

# ── Input boxes ──
ax.text(2.5, 9.2, 'Single repr. $\\mathbf{s}_i$\n(from Evoformer)', fontsize=12,
        ha='center', va='center',
        bbox=dict(**box_props, facecolor=c_input, edgecolor='#4292c6'))
ax.text(5.5, 9.2, 'Pair repr. $\\mathbf{z}_{ij}$\n(from Evoformer)', fontsize=12,
        ha='center', va='center',
        bbox=dict(**box_props, facecolor=c_input, edgecolor='#4292c6'))
ax.text(8.5, 9.2, 'Initial frames\n$T_i^{(0)} = (\\mathbf{I}, \\mathbf{0})$', fontsize=12,
        ha='center', va='center',
        bbox=dict(**box_props, facecolor=c_input, edgecolor='#4292c6'))

# Arrows from inputs
ax.annotate('', xy=(5.0, 7.6), xytext=(2.5, 8.55),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#333333'))
ax.annotate('', xy=(5.5, 7.6), xytext=(5.5, 8.55),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#333333'))
ax.annotate('', xy=(6.5, 7.6), xytext=(8.5, 8.55),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#333333'))

# ── Iteration loop box ──
loop_rect = mpatches.FancyBboxPatch((1.0, 1.5), 10.5, 6.3,
                                     boxstyle='round,pad=0.3',
                                     facecolor='#f7f7f7', edgecolor=c_loop,
                                     linewidth=2.5, linestyle='--')
ax.add_patch(loop_rect)
ax.text(11.2, 7.5, 'Repeat\n8 times\n(shared\nweights)', fontsize=11,
        ha='center', va='center', color=c_loop, style='italic')

# ── IPA block ──
y_ipa = 6.8
ax.text(5.5, y_ipa, 'Invariant Point Attention (IPA)', fontsize=13,
        ha='center', va='center',
        bbox=dict(**box_props, facecolor=c_ipa, edgecolor='#d94801'))
ax.text(5.5, y_ipa - 0.55, 'Sequence attn + Pair bias + Point attn', fontsize=10,
        ha='center', va='center', color='#666666')

# Arrow IPA -> Transition
ax.annotate('', xy=(5.5, 5.45), xytext=(5.5, 6.15),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#333333'))

# ── Transition block ──
y_ffn = 5.0
ax.text(5.5, y_ffn, 'Transition (FFN)', fontsize=13,
        ha='center', va='center',
        bbox=dict(**box_props, facecolor=c_ffn, edgecolor='#238b45'))

# Arrow Transition -> Backbone Update and Torsion (fork)
ax.annotate('', xy=(3.8, 3.55), xytext=(5.0, 4.55),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#333333'))
ax.annotate('', xy=(7.2, 3.55), xytext=(6.0, 4.55),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#333333'))

# ── Backbone Update ──
y_bb = 3.0
ax.text(3.5, y_bb, 'Backbone Update\n$T_i^{(l+1)} = T_i^{(l)} \\circ \\Delta T_i$',
        fontsize=12, ha='center', va='center',
        bbox=dict(**box_props, facecolor=c_bb, edgecolor='#6a51a3'))

# ── Torsion Prediction ──
ax.text(8.0, y_bb, 'Torsion Prediction\n$(\\sin\\chi_k, \\cos\\chi_k)$',
        fontsize=12, ha='center', va='center',
        bbox=dict(**box_props, facecolor=c_torsion, edgecolor='#cb181d'))

# Loop-back arrow from backbone update back to IPA
ax.annotate('', xy=(2.0, 6.8), xytext=(2.0, 3.0),
            arrowprops=dict(arrowstyle='->', lw=2, color=c_loop,
                           connectionstyle='arc3,rad=0.3'))
ax.text(1.4, 5.0, 'Updated\nframes', fontsize=10, ha='center', color=c_loop, rotation=90)

# ── Output ──
ax.annotate('', xy=(5.5, 0.6), xytext=(3.5, 2.2),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#333333'))
ax.annotate('', xy=(5.5, 0.6), xytext=(8.0, 2.2),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#333333'))

ax.text(5.5, 0.3, 'All-atom coordinates', fontsize=13,
        ha='center', va='center',
        bbox=dict(**box_props, facecolor=c_output, edgecolor='#756bb1'))

ax.set_title('The Structure Module Pipeline', fontsize=15, pad=15)
plt.tight_layout()
plt.show()

---
## 9. Summary and Key Takeaways

In this notebook we dissected the **Structure Module**, the component of AlphaFold2 that converts abstract Evoformer representations into 3D atomic coordinates.

### Core ideas

1. **Rigid-body frames in SE(3).** Each residue's backbone is parameterized as a frame $T_i = (\mathbf{R}_i, \mathbf{t}_i)$. All structural operations (composition, application, inversion) follow the group law of SE(3), ensuring mathematical consistency.

2. **Invariant Point Attention (IPA).** The central innovation of the Structure Module. IPA combines three sources of information:
   - Standard sequence-based query-key-value attention,
   - Pair representation bias,
   - **Point attention**, where 3D points are projected from local residue frames into a shared global frame and compared via Euclidean distance.
   
   The distance-based component makes IPA spatially aware: residues that are close in 3D attend more strongly to each other, regardless of their sequence separation.

3. **SE(3)-equivariance by construction.** By expressing all updates in local frames and computing distances that are invariant to global transformations, the Structure Module guarantees that a global rotation/translation of the input produces a correspondingly transformed output. This is not enforced via data augmentation or loss penalties -- it is an exact, architectural property.

4. **Iterative refinement.** Starting from all frames at the identity (collapsed at the origin), the Structure Module runs 8 iterations of IPA + backbone update + torsion prediction, progressively "folding" the protein. Weight sharing across iterations makes this efficient.

5. **From frames to atoms.** Backbone frames, combined with predicted torsion angles $(\sin\chi, \cos\chi)$, determine all heavy-atom positions through a chain of rigid-body transformations applied to ideal bond geometry.

### Preview: Notebook 7

We have now covered all major architectural components of AlphaFold2. In the next notebook, we turn to the question of **training**: what loss functions does AlphaFold2 use, and how are they designed to produce accurate structures? We will examine the FAPE (Frame Aligned Point Error) loss, auxiliary losses, and the confidence-calibrated pLDDT metric.